# Prepare Crunchbase data for Mission Studio / Mission Radar

In [1]:
import pandas as pd

from src import PROJECT_DIR, logging

from discovery_utils.getters import crunchbase
from discovery_utils.utils.io import safe_yaml_load
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts
)

from discovery_utils.utils.llm import batch_check


PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}"

In [2]:
CB = crunchbase.CrunchbaseGetter()

2025-04-14 10:16:30,884 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-04-14 10:16:30,999 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-04-14


In [3]:
CONFIG_NAMES = [
    "bioenergy",
    "biomass_heating",
    "built_environment",
    "ccus",
    # "decarbonisation_general",
    "district_heating",
    "energy_efficiency",
    "energy_grid",
    # "energy_storage",
    "geothermal_energy",
    "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    # "renewables_general",
    "solar_thermal",
    # "solar",
    # "wind"
]

## Run the LLM checks

In [ ]:
def get_config_dict(config_name: str) -> dict:
    """Find companies in a specific category from a config file"""
    config_path = str(PROJECT_DIR / f"notebooks/{PROJECT_NAME}/config_{config_name}.yaml")
    return safe_yaml_load(open(config_path))

def get_companies_from_config(config: dict) -> pd.DataFrame:
    """Get companies from a config file"""
    category_name = config["search_recipe"]["category_name"]
    return CB.get_companies_in_nesta_categories("topic_labels", [category_name])

async def check_relevance(selected_df: pd.DataFrame, config_name: str, config: dict) -> None:
    """Check relevance of the selected companies"""
    selected_texts_df = CB.get_organisation_text(selected_df)
    check_data = dict(zip(selected_texts_df['id'], selected_texts_df['text']))
    system_message = batch_check.generate_relevance_check_system_message(config)

    fields = [
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    ]

    processor = batch_check.LLMProcessor(
        output_path=str(OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=10, sleep_time=0.5)    

In [5]:
async def check_all_configs(config_names) -> None:
    """Check relevance for all config files"""
    for config_name in config_names:
        logging.info(f"Checking relevance for {config_name}")
        config = get_config_dict(config_name)
        selected_df = get_companies_from_config(config)
        await check_relevance(selected_df, config_name, config)
        

In [6]:
await check_all_configs(CONFIG_NAMES)

2025-04-03 17:22:31,964 - root - INFO - Checking relevance for bioenergy
2025-04-03 17:22:31,968 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/organizations_full.parquet
2025-04-03 17:22:32,097 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-03 17:22:32,126 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-03 17:22:32,128 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-03 17:22:32,136 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-03 17:22:32,139 - botocore.httpchecksum - INFO - Skipping checksum validati

## Spot check the results

In [7]:
sheet_id = "1m9_tKyJDaSy2vDWxYVP_9HlfBbGysQUV-xrb1FW3vok"

In [15]:
import pandas as pd
from discovery_utils.utils import google

n_samples = 10
final_cols = ["theme", "id", "name", "text", "total_funding_gbp", "cb_url", "homepage_url", "is_relevant"]

llm_checks_df = []

for config_name in CONFIG_NAMES:
    # read a jsonl file
    llm_check_df = pd.read_json(OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl", lines=True)
    # get company descriptions
    org_texts = CB.get_organisation_text(llm_check_df)
    # sample n_samples from each group
    llm_check_df = (
        llm_check_df
        .merge(CB.organisations_enriched, left_on="id", right_on="id", how="left")
        .merge(org_texts, left_on="id", right_on="id", how="left")
        .assign(theme=config_name)
        .groupby("is_relevant")[final_cols]
        .apply(lambda df: df.sample(n_samples) if len(df) > n_samples else df)
        .reset_index(drop=True)
    )[final_cols]
    llm_checks_df.append(llm_check_df)

llm_checks_df = pd.concat(llm_checks_df, ignore_index=True)


In [ ]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_check": llm_checks_df})
google.format_gsheet(sheet_id, "crunchbase_check", freeze_cols=2)

2025-03-19 06:59:34,988 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]
2025-03-19 06:59:36,006 - root - INFO - Uploading DataFrame to sheet: crunchbase_check
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Library/Caches/pypoetry/virtualenvs/discovery-mission-radar-prototyping-ejbE0IFh-py3.11/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-03-19 06:59:47,629 - root - INFO - Upload completed successfully.
2025-03-19 06:59:48,243 - root - INFO - Connected to Google Sheet: Mission Radar spot checks [2025-03-19]
